# Picking a model when you can't see the loss surface

In notebook 2 we fit a *cubic* to noisy cubic data. That was cheating: we **knew** the true function family, so picking the model was trivial. We could also still draw the loss surface in two notebooks ago — two parameters lived comfortably in 3-D.

In the real world, neither of those luxuries survives:

- The true function is **unknown**. All you ever see is a table of `(x, y)` observations.
- Any interesting model has **more than two parameters**, so the loss landscape lives in a space you cannot visualize.

So how do you choose a model? You **guess** — make a flexible-enough assumption, fit it, see what happens, revise. Every practitioner does this. This notebook walks through the failure mode of that guess-and-fit loop, and the standard tool we use to fix it: **regularization**.

The plan:

1. Look at the same dataset from notebook 2, but as a raw **table of observations** — no overlaid ground truth.
2. Pretend we don't know it's a cubic. Make the assumption *"a polynomial of degree 7 should be flexible enough"*.
3. Fit it with Adam, look at the result.
4. Diagnose: the model **overfits** — it bends through noise instead of capturing signal.
5. Introduce **weight decay** as the simplest form of regularization and re-fit.

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import torch
import torch.nn as nn

pio.renderers.default = "notebook"

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
print(f"PyTorch: {torch.__version__}")
print(f"Device:  {device}")

## The observations

Same data-generating process as notebook 2 — a cubic plus Gaussian noise — but for this notebook **pretend you've never seen the line `y_clean = a_true·x³ + b_true·x² + c_true·x + d_true`**. All you have is the table below. Stare at it: you can probably tell *something* is curving, but you can't tell at a glance whether the underlying function is quadratic, cubic, exponential, or something else.

To make the choice of model harder (and overfitting easier to see), we'll use **only a small training subset** of these observations — 15 randomly chosen points — and hold out the rest as evaluation. That mirrors real life: collecting more data is usually expensive, so you do the best you can with what you have.

In [ ]:
rng = np.random.default_rng(seed=0)

# Same generating process as notebook 2 (cubic + Gaussian noise)
a_true, b_true, c_true, d_true = 0.5, -1.2, -0.7, 1.0
sigma_noise = 1.5

n_samples = 60
x_np = np.linspace(-3.5, 3.5, n_samples)
y_clean = a_true * x_np**3 + b_true * x_np**2 + c_true * x_np + d_true
y_np    = y_clean + rng.normal(0.0, sigma_noise, size=n_samples)

# Pick a small training subset; the rest is held out.
n_train = 15
train_idx = np.sort(rng.choice(n_samples, size=n_train, replace=False))
val_idx   = np.array([i for i in range(n_samples) if i not in set(train_idx)])

x_train, y_train = x_np[train_idx], y_np[train_idx]
x_val,   y_val   = x_np[val_idx],   y_np[val_idx]

# Display the training observations as a clean DataFrame
obs = pd.DataFrame({"i": train_idx, "x": x_train.round(3), "y": y_train.round(3)})
print(f"{n_train} training observations (out of {n_samples} total):")
obs.reset_index(drop=True)

In [ ]:
# Scatter the data (without overlaying the true cubic — we're pretending we don't know it).
scatter_fig = go.Figure()
scatter_fig.add_trace(go.Scatter(
    x=x_train, y=y_train, mode="markers", name="training (n=15)",
    marker=dict(size=10, color="dodgerblue", line=dict(color="black", width=1)),
))
scatter_fig.add_trace(go.Scatter(
    x=x_val, y=y_val, mode="markers", name="held out",
    marker=dict(size=6, color="lightgray", opacity=0.7),
))
scatter_fig.update_layout(
    title="What's the underlying function? (15 training points in blue, 45 held out in gray)",
    xaxis_title="x", yaxis_title="y",
    template="plotly_white", width=800, height=440,
)
scatter_fig.show()

## The modeling assumption: polynomial of degree 7

We don't know the underlying function. We **can't** plot the loss landscape of any reasonable model to look for its minimum — even a 4-parameter cubic puts us in 5-D, and anything richer goes further out of reach. We have to commit to an assumption and see how it fares.

Let's pick a flexible default: a **polynomial of degree 7**.

$$f(x;\, \boldsymbol{\theta}) \;=\; \theta_0 \;+\; \theta_1\, x \;+\; \theta_2\, x^2 \;+\; \theta_3\, x^3 \;+\; \theta_4\, x^4 \;+\; \theta_5\, x^5 \;+\; \theta_6\, x^6 \;+\; \theta_7\, x^7$$

Eight parameters. The reasoning is *"surely 8 parameters can capture whatever curve is hiding in there"* — and that's true, but it's also the trap. With only 15 training points, an 8-parameter polynomial has so much flexibility that it can pass arbitrarily close to *every* training point — including the noise on top of each one. That's overfitting.

To make the numerics stable, we'll normalize the input by dividing by `x_max = 3.5` before raising to powers (so the largest input magnitudes are around 1.0 instead of 3.5⁷ ≈ 6400). It doesn't change the function family at all, only the scale of the learned coefficients.

In [ ]:
X_MAX = 3.5  # max |x| in the data, used to scale powers down to roughly [-1, 1]

class Polynomial(nn.Module):
    """f(x) = sum_k theta_k * (x / X_MAX)^k for k in 0..degree."""
    def __init__(self, degree=7):
        super().__init__()
        self.degree = degree
        self.coeffs = nn.Parameter(torch.zeros(degree + 1))

    def forward(self, x):
        xn = x / X_MAX                                           # (B,)
        powers = torch.stack([xn**k for k in range(self.degree + 1)], dim=-1)  # (B, degree+1)
        return powers @ self.coeffs                              # (B,)


# Move training data onto the device
x_train_t = torch.tensor(x_train, dtype=torch.float32, device=device)
y_train_t = torch.tensor(y_train, dtype=torch.float32, device=device)
x_val_t   = torch.tensor(x_val,   dtype=torch.float32, device=device)
y_val_t   = torch.tensor(y_val,   dtype=torch.float32, device=device)


def train_polynomial(weight_decay=0.0, n_steps=4000, lr=0.02, degree=7, seed=0):
    """Train a polynomial of the given degree with Adam, optionally with L2 weight decay."""
    torch.manual_seed(seed)
    model = Polynomial(degree=degree).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()

    train_losses, val_losses = [], []
    for step in range(n_steps):
        # train
        y_pred = model(x_train_t)
        loss = loss_fn(y_pred, y_train_t)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

        # val (no grad, no weight-decay accounting)
        with torch.no_grad():
            val_losses.append(loss_fn(model(x_val_t), y_val_t).item())

    return model, np.array(train_losses), np.array(val_losses)

## Fit it — no regularization yet

Standard PyTorch training loop, Adam, MSE loss. We track both the **training loss** (on our 15 points) and the **validation loss** (on the 45 held-out points). The contrast between the two is what diagnoses overfitting.

In [ ]:
model_nowd, train_losses_nowd, val_losses_nowd = train_polynomial(weight_decay=0.0)

print(f"Final train loss = {train_losses_nowd[-1]:.4f}")
print(f"Final val   loss = {val_losses_nowd[-1]:.4f}   ← much bigger? that's overfitting")
print()
print("Learned coefficients (θ_0 … θ_7):")
for k, theta in enumerate(model_nowd.coeffs.detach().cpu().numpy()):
    print(f"  θ_{k}  = {theta:+.3f}")

In [ ]:
x_dense = np.linspace(-4.0, 4.0, 400)  # slightly past training range, to expose edge wiggles
y_dense_true = a_true * x_dense**3 + b_true * x_dense**2 + c_true * x_dense + d_true

with torch.no_grad():
    y_dense_fit_nowd = model_nowd(torch.tensor(x_dense, dtype=torch.float32, device=device)).cpu().numpy()

fit_fig = go.Figure()
fit_fig.add_trace(go.Scatter(x=x_val, y=y_val, mode="markers", name="held out",
                             marker=dict(size=6, color="lightgray", opacity=0.7)))
fit_fig.add_trace(go.Scatter(x=x_train, y=y_train, mode="markers", name="training (n=15)",
                             marker=dict(size=10, color="dodgerblue",
                                         line=dict(color="black", width=1))))
fit_fig.add_trace(go.Scatter(x=x_dense, y=y_dense_true, mode="lines",
                             name="true cubic (the secret)",
                             line=dict(color="crimson", width=2, dash="dash")))
fit_fig.add_trace(go.Scatter(x=x_dense, y=y_dense_fit_nowd, mode="lines",
                             name="poly-7 fit (no regularization)",
                             line=dict(color="orange", width=3)))
fit_fig.update_layout(
    title="Polynomial-of-degree-7 fit to 15 training points",
    xaxis_title="x", yaxis_title="y",
    template="plotly_white", width=850, height=460,
)
fit_fig.show()

# Loss curves
loss_fig = go.Figure()
loss_fig.add_trace(go.Scatter(y=train_losses_nowd, mode="lines",
                              name="train MSE", line=dict(color="dodgerblue", width=2)))
loss_fig.add_trace(go.Scatter(y=val_losses_nowd, mode="lines",
                              name="val MSE", line=dict(color="crimson", width=2)))
loss_fig.update_layout(
    title="Training vs validation loss (no regularization)",
    xaxis_title="step", yaxis_title="MSE",
    yaxis_type="log",
    template="plotly_white", width=850, height=360,
)
loss_fig.show()

## What you should see — and what it means

Two signs of **overfitting**:

1. **The orange curve bends through every blue training point** but takes off wildly on either edge of the training range. That's the polynomial spending its degree-7 flexibility on memorizing the noise in those 15 specific samples, not on capturing the underlying smooth curve.
2. **The validation loss is much higher than the training loss** — the gap between the blue (train) and red (val) curves on the loss plot. The model nailed the training data and is bad at generalizing.

Look at the coefficient magnitudes printed above. They're often dozens or hundreds in absolute value. That's the signature of overfitting in polynomial regression: tiny changes in the data direction get amplified into huge swings, achieved by piling up large coefficients with opposing signs that nearly cancel. The model has the right *fit*, but not the right *shape*.

The fundamental tension is:

- We don't *want* less flexibility — the model needs to be flexible enough to capture whatever the true function is.
- We *do* want a way to tell the optimizer **"prefer simpler solutions when several explain the data equally well"**.

That's exactly what **regularization** does.

## Regularization: weight decay (L2)

The simplest form is **L2 regularization**, also called **weight decay**. We add a penalty term proportional to the squared norm of the parameters to the loss:

$$\mathcal{L}_\text{reg}(\boldsymbol{\theta}) \;=\; \underbrace{\text{MSE}(y, \hat{y})}_{\text{data fit}} \;+\; \lambda \cdot \underbrace{\sum_k \theta_k^2}_{\text{penalty on big weights}}$$

Now the optimizer faces a tradeoff: it can keep shrinking the data-fit term by adding extra polynomial flexibility, but every time it does, the penalty term grows. So it learns to use as little parameter magnitude as possible to explain the data — which means **simpler-shaped fits**.

In PyTorch this is one extra kwarg to the optimizer:

```python
torch.optim.Adam(model.parameters(), lr=0.02, weight_decay=λ)
```

That's it. Same training loop, same model, same data. Just one number tuned. Let's try $\lambda = 0.05$.

In [ ]:
LAMBDA = 0.05

model_wd, train_losses_wd, val_losses_wd = train_polynomial(weight_decay=LAMBDA)

print(f"Final train loss (no wd) = {train_losses_nowd[-1]:.4f}    val loss = {val_losses_nowd[-1]:.4f}")
print(f"Final train loss (wd={LAMBDA}) = {train_losses_wd[-1]:.4f}    val loss = {val_losses_wd[-1]:.4f}")
print()
print("Learned coefficients with weight decay:")
for k, theta in enumerate(model_wd.coeffs.detach().cpu().numpy()):
    print(f"  θ_{k}  = {theta:+.3f}")

In [ ]:
with torch.no_grad():
    y_dense_fit_wd = model_wd(torch.tensor(x_dense, dtype=torch.float32, device=device)).cpu().numpy()

compare_fig = go.Figure()
compare_fig.add_trace(go.Scatter(x=x_val, y=y_val, mode="markers", name="held out",
                                 marker=dict(size=6, color="lightgray", opacity=0.7)))
compare_fig.add_trace(go.Scatter(x=x_train, y=y_train, mode="markers", name="training (n=15)",
                                 marker=dict(size=10, color="dodgerblue",
                                             line=dict(color="black", width=1))))
compare_fig.add_trace(go.Scatter(x=x_dense, y=y_dense_true, mode="lines",
                                 name="true cubic",
                                 line=dict(color="crimson", width=2, dash="dash")))
compare_fig.add_trace(go.Scatter(x=x_dense, y=y_dense_fit_nowd, mode="lines",
                                 name="poly-7, λ = 0 (overfit)",
                                 line=dict(color="orange", width=2)))
compare_fig.add_trace(go.Scatter(x=x_dense, y=y_dense_fit_wd, mode="lines",
                                 name=f"poly-7, λ = {LAMBDA} (regularized)",
                                 line=dict(color="seagreen", width=3)))
compare_fig.update_layout(
    title="Same model, same data — only weight decay differs",
    xaxis_title="x", yaxis_title="y",
    template="plotly_white", width=850, height=460,
)
compare_fig.show()

loss_compare = go.Figure()
loss_compare.add_trace(go.Scatter(y=val_losses_nowd, mode="lines",
                                  name="val MSE, λ = 0",
                                  line=dict(color="orange", width=2)))
loss_compare.add_trace(go.Scatter(y=val_losses_wd, mode="lines",
                                  name=f"val MSE, λ = {LAMBDA}",
                                  line=dict(color="seagreen", width=2)))
loss_compare.update_layout(
    title="Validation loss: regularization keeps the generalization gap closed",
    xaxis_title="step", yaxis_title="validation MSE",
    yaxis_type="log",
    template="plotly_white", width=850, height=360,
)
loss_compare.show()

## Takeaways

- **Without ground truth or a visualizable loss surface, model choice is a guess.** You pick something flexible, fit it, and revise based on what you observe — especially the gap between training and validation loss.
- **More parameters than data points = potential overfitting.** The model has enough capacity to memorize noise, and it will if you let it.
- **Regularization is the standard fix.** Weight decay (L2) adds a penalty for large parameter values to the loss, biasing the optimizer toward simpler solutions. In PyTorch it's a single optimizer kwarg.
- **It generalizes far beyond polynomials.** Every neural network you train from here on — including the MLP, the RNN in this series, and GPT itself — uses weight decay (or a related technique like dropout, gradient clipping, or label smoothing) to fight overfitting. The principle is identical: shape the loss so the optimizer prefers "simpler" models when several explain the data equally well.

Things to try yourself:

- Bump `LAMBDA` to `1.0` or `5.0`. What happens to the fit? (Spoiler: **underfitting** — the penalty becomes louder than the data.)
- Drop `LAMBDA` to `1e-4`. Is overfitting back?
- Increase `n_train` from `15` to `40`. Does the unregularized polynomial still overfit, or does the extra data alone fix it?
- Change `degree=7` to `degree=15` in the no-regularization run. Even with 15 training points and `weight_decay=0`, can you get a smooth fit by adjusting `lr` and step count alone? (No — without regularization, you eventually overfit; more capacity just makes it faster.)